# 02 — PDE10A EDA & Baseline Models

**Dataset**: 1162 PDE10A inhibitors with pIC50 values  
**Focus**: EDA (pIC50 distribution, binding modes, temporal coverage, split strategies); then baseline ML models across all 7 pre-defined split strategies (temporal 2011–2013, 3 chemotype-based, random).  
**Reuses**: `src.splitting`, `src.features`, `src.models`, `src.plotting`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.splitting import get_split, list_split_cols, SPLIT_COLS
from src.eda import smiles_validity_report, missing_value_report
from src.features import morgan_fingerprints, rdkit_descriptors
from src.models import get_baseline_models, evaluate_model
from src.plotting import pred_vs_actual_grid

DATA_PATH = '../data/raw/10822_2022_478_MOESM2_ESM.csv'
SMILES_COL = 'SMILES'
TARGET_COL = 'pic50'
SEED = 42


## Part 1 — Exploratory Data Analysis

### 1.1 Load & Inspect

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
df.head(3)


### 1.2 SMILES Validity

In [ ]:
report = smiles_validity_report(df[SMILES_COL].tolist())
print(report)


### 1.3 Duplicate Check

In [ ]:
from rdkit import Chem

n_raw_dupes = df[SMILES_COL].duplicated().sum()
canonical = df[SMILES_COL].apply(lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)))
n_canon_dupes = canonical.duplicated().sum()

print(f'Raw SMILES duplicates:       {n_raw_dupes}')
print(f'Canonical SMILES duplicates: {n_canon_dupes}')


### 1.4 pIC50 Distribution

In [ ]:
from scipy.stats import skew, kurtosis

pic50 = df[TARGET_COL].dropna()
print(f'Count:    {len(pic50)}')
print(f'Mean:     {pic50.mean():.3f}')
print(f'Std:      {pic50.std():.3f}')
print(f'Min/Max:  {pic50.min():.3f} / {pic50.max():.3f}')
print(f'Skewness: {skew(pic50):.3f}')
print(f'Kurtosis: {kurtosis(pic50):.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(pic50, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('pIC50')
ax.set_ylabel('Count')
ax.set_title('PDE10A pIC50 Distribution')
plt.tight_layout()
plt.show()


### 1.5 Binding Mode Classes

In [ ]:
counts = df['binding_mode_class'].value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(8, 3))
counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Count')
ax.set_title('Binding Mode Class Distribution')
plt.tight_layout()
plt.show()


### 1.6 Temporal Coverage

In [ ]:
df['year'] = pd.to_datetime(df['pic50_date'], format='%d-%b-%Y').dt.year
year_counts = df['year'].value_counts().sort_index()
print(year_counts)

fig, ax = plt.subplots(figsize=(8, 3))
year_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Year')
ax.set_ylabel('Count')
ax.set_title('Compounds by Year (pic50_date)')
plt.tight_layout()
plt.show()


### 1.7 Split Strategy Overview

Each split column assigns rows to `train`, `test`, or `val`. `val` rows and empty rows are excluded from modelling (SYNC-007).

In [ ]:
rows_overview = []
for col in SPLIT_COLS:
    vc = df[col].value_counts()
    empty = (df[col] == '').sum() + df[col].isna().sum()
    rows_overview.append({
        'split': col,
        'train': int(vc.get('train', 0)),
        'test': int(vc.get('test', 0)),
        'val': int(vc.get('val', 0)),
        'empty/NaN': int(empty),
    })
split_overview = pd.DataFrame(rows_overview).set_index('split')
display(split_overview)


### 1.8 Chemical Space

In [ ]:
desc = rdkit_descriptors(df[SMILES_COL].tolist())
print(desc.describe().round(2))

ro5_violations = (
    (desc['MW'] > 500).astype(int)
    + (desc['LogP'] > 5).astype(int)
    + (desc['HBD'] > 5).astype(int)
    + (desc['HBA'] > 10).astype(int)
)
print(f'\nLipinski Ro5 — compounds with >=2 violations: {(ro5_violations >= 2).sum()} / {len(desc)}')


### 1.9 EDA Conclusions

- **1162 compounds**, single target (`pic50`), no missing values
- SMILES validity and duplicate counts as printed above
- `pic50` distribution: unimodal, centred ~7–9 (potent PDE10A inhibitors)
- 3 binding mode classes; `aryl_c1_amide_c2_hetaryl` is the largest group
- Temporal range: 2010–2013; temporal splits reflect this chronology
- `random_split`: most complete (0 empty rows); `temporal_2011_split`: 219 empty rows
- Most compounds satisfy Lipinski Ro5 (expected for CNS-penetrant PDE10A inhibitors)


---

## Part 2 — Featurization & Baseline Models

Same 5 models as ADME Phase 1: LinearRegression, BayesianRidge, RandomForest, XGBoost, LightGBM.  
Morgan fingerprints (ECFP4, radius=2, 2048 bits) computed **once** for all 1162 compounds and indexed per split — no recomputation.  
Metrics: R² and RMSE on the `test` partition of each of the 7 split strategies.

### 2.1 Featurize All Compounds (ECFP4)

In [ ]:
print('Computing ECFP4 fingerprints for all 1162 compounds...')
X_all = morgan_fingerprints(df[SMILES_COL].tolist())
y_all = df[TARGET_COL].values
print(f'X_all shape: {X_all.shape}')
print(f'y_all shape: {y_all.shape}')


### 2.2 Build Train/Test Splits

In [ ]:
split_data = {}
for split_col in list_split_cols(df):
    train_df, test_df = get_split(df, split_col)
    split_data[split_col] = (
        X_all[train_df.index],
        X_all[test_df.index],
        y_all[train_df.index],
        y_all[test_df.index],
    )
    print(f'{split_col}: train={len(train_df)}, test={len(test_df)}')


### 2.3 Model Training & Evaluation

In [ ]:
rows_results = []
preds_store = {}

for split_col, (X_tr, X_te, y_tr, y_te) in split_data.items():
    preds_store[split_col] = {}
    for name, model in get_baseline_models().items():
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        preds_store[split_col][name] = (y_te, y_pred)
        metrics = evaluate_model(model, X_te, y_te, y_pred=y_pred)
        rows_results.append({'split': split_col, 'model': name, **metrics})

results_df = pd.DataFrame(rows_results)
print(f'Results: {results_df.shape[0]} rows (expected 35 = 7 splits x 5 models)')
results_df.head(10)

### 2.4 Results Summary

In [ ]:
r2_pivot = results_df.pivot(index='split', columns='model', values='R2').round(3)
rmse_pivot = results_df.pivot(index='split', columns='model', values='RMSE').round(3)

print('R\u00b2 by split and model:')
display(r2_pivot)
print('\nRMSE by split and model:')
display(rmse_pivot)


### 2.5 Cross-Split Comparison

R² per model across all 7 split strategies. Scientific question: does the choice of split inflate or deflate reported R²? Random splits typically give the most optimistic R²; temporal and chemotype splits are harder tests of generalisation.

In [ ]:
model_names = ['LinearRegression', 'BayesianRidge', 'RandomForest', 'XGBoost', 'LightGBM']
split_keys = list(split_data.keys())
split_labels = [s.replace('_split', '') for s in split_keys]

x = np.arange(len(split_labels))
width = 0.15

fig, ax = plt.subplots(figsize=(14, 5))
for i, model in enumerate(model_names):
    r2_vals = [
        results_df[(results_df['split'] == s) & (results_df['model'] == model)]['R2'].values[0]
        for s in split_keys
    ]
    ax.bar(x + i * width, r2_vals, width, label=model)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(split_labels, rotation=30, ha='right')
ax.set_ylabel('R\u00b2')
ax.set_title('R\u00b2 by Split Strategy and Model \u2014 PDE10A')
ax.legend(loc='upper right', fontsize=8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()


### 2.6 Predicted vs Actual — random_split (reference)

In [ ]:
fig = pred_vs_actual_grid(
    preds_store['random_split'],
    title='PDE10A pIC50 \u2014 random_split (predicted vs actual)'
)
plt.show()
